# Canada Food CPI — CFPR Replica Experiment

**Core question:** of the methods evaluated here, which produces the most
calibrated per-category forecast for the next Canada's Food Price Report?

This notebook runs a multi-predictor backtest against a configurable CFPR spec
and answers that question via per-category model selection on CRPS and MAPE.
Set `EXPERIMENT_CONFIG` in the config cell to choose the scope:
`mini_single` (1 target, 6 origins — default), `mini_recent` (9 targets, 6 origins),
or `full` (9 targets, 16 origins — canonical CFPR backtest).

**What's here:**

1. Data overview — nine food CPI sub-indices at a glance.
2. Spec + predictors — loaded from YAML; backtest results cached on disk.
3. Qualitative check — trajectory fans and avg/avg YoY grid.
4. Model selection — CRPS and MAPE per category; per-category winner table.
5. Protected eval — budget-enforced holdout (`full` config only).


---
## 1. Setup

The heavy lifting lives in helper modules alongside this notebook:

- `data.py`      registers the 9 StatCan series on a `DataService`.
- `analysis.py`  flattens results to DataFrames and computes avg/avg YoY.
- `plots.py`     renders the figures the CFPR audience expects.

Backtest specs (all under `reference_specs/food_cpi/`):

| Spec file | Tasks | Origins | Notes |
|---|---|---|---|
| `food_cpi_single_mini_backtest.yaml` | 1 (food overall) | 6 (2019–2024) | Fast dev/smoke-test |
| `food_cpi_recent_backtest.yaml` | 9 | 6 (2019–2024) | Recent regimes only |
| `food_cpi_cfpr_backtest.yaml` | 9 | 16 (2009–2024) | Canonical CFPR backtest |
| `food_cpi_cfpr_eval.yaml` | 9 | 4 (2021–2024) | Protected eval, `full` only |

Run the data fetch once if you haven't:

```bash
uv run python scripts/fetch_cpi.py
```


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from dotenv import load_dotenv


warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve().parents[1]
load_dotenv(ROOT / ".env")

from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    MultiTargetEvalSpec,
    cached_multi_backtest,
    describe_spec,
)
from aieng.forecasting.evaluation.eval import EvalTracker
from aieng.forecasting.methods import DartsAutoARIMAPredictor, LastValuePredictor
from food_price_forecasting.analysis import compute_ape_long, compute_avgyoy, compute_mape, summarize_crps
from food_price_forecasting.data import CATEGORY_LABELS, build_food_cpi_service
from food_price_forecasting.plots import (
    plot_avgyoy_grid,
    plot_food_cpi_small_multiples,
    plot_mape_by_category,
    plot_trajectory_fan,
)


STATCAN_CACHE = ROOT / "data" / "statcan"
PREDICTIONS_DIR = ROOT / "data" / "predictions"
EVAL_RUNS_PATH = ROOT / "data" / "eval_runs.yaml"
SPECS_DIR = ROOT / "reference_specs" / "food_cpi"

svc = build_food_cpi_service(cache_dir=STATCAN_CACHE)
print(f"Registered {len(CATEGORY_LABELS)} food CPI series.")

In [ ]:
# ── Experiment configuration ──────────────────────────────────────────────────
# Set EXPERIMENT_CONFIG to control which CFPR backtest runs throughout this
# notebook. All downstream cells adapt automatically.
#
#   "mini_single"  1 target (food overall) × 6 recent origins (Jul 2019–2024)
#                  ~6 agent calls — fastest; ideal for agent development
#   "mini_recent"  9 targets × 6 recent origins (Jul 2019–2024)
#                  ~54 agent calls — covers COVID/inflation regimes
#   "full"         9 targets × 16 origins (Jul 2009–2024)
#                  ~144 agent calls — canonical CFPR backtest

EXPERIMENT_CONFIG = "mini_single"

_CONFIGS = {
    "mini_single": ("food_cpi_single_mini_backtest.yaml", None),
    "mini_recent": ("food_cpi_recent_backtest.yaml",      None),
    "full":        ("food_cpi_cfpr_backtest.yaml",        "food_cpi_cfpr_eval.yaml"),
}
_BACKTEST_SPEC_FILE, _EVAL_SPEC_FILE = _CONFIGS[EXPERIMENT_CONFIG]

print(f"Config: {EXPERIMENT_CONFIG!r}  →  {_BACKTEST_SPEC_FILE}")

---
## 2. Data exploration

A single figure is plenty — the nine sub-indices track each other closely with
a clear post-2020 acceleration.

In [ ]:
fig, _ = plot_food_cpi_small_multiples(svc)
plt.show()

---
## 3. The backtest spec

The backtest spec is loaded from YAML so the spec (not the notebook) is the
source of truth.  `describe_spec()` renders a plain-text summary suitable for
print, prompts, or documentation.

In [ ]:
with (SPECS_DIR / _BACKTEST_SPEC_FILE).open() as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))

print(describe_spec(backtest_spec, data_service=svc))
print(
    f"\nTasks: {len(backtest_spec.tasks)}  Window: {backtest_spec.start.date()} → {backtest_spec.end.date()}  Stride: {backtest_spec.stride}"
)

---
## 4. Predictors

Eight predictors across four groups, each implementing the same `Predictor`
API against the configured backtest spec.  Swap in your own subclass or add
to `all_predictors` to run it through the full evaluation loop.

| Group | Predictor | Notes |
|---|---|---|
| Baselines | `LastValuePredictor` | Repeats last observed value; hard to beat at short horizons. |
| Baselines | `DartsAutoARIMAPredictor` | Auto-ARIMA via Darts; fits at each origin. |
| LLMP ×2 | `ContinuousLLMPredictor` | Direct LLM call; structured JSON trajectory; no tools or search. |
| Agent ×2 (search off) | `AgentPredictor` | ADK agent with food CPI instruction; historical-backtest-safe. |
| Agent ×2 (search on) | `AgentPredictor` | Same agent with `context_agent` (bounded Google Search) enabled. |

### Agent configuration: identity vs. role

`AgentConfig` captures the agent's **identity** — instruction, model, capability
toggles.  `AgentPredictor` captures the agent's **role** — which output schema
it must satisfy in this experiment.  The same config can be reused across roles.

**One-liner (default settings):**

```python
predictor = build_food_price_agent_predictor(model="gemini-3-flash-preview")
```

**Explicit construction (shows the identity/role split):**

```python
from aieng.forecasting.methods.agentic import AgentPredictor, ContinuousAgentForecastOutput
from food_price_forecasting.analyst_agent import (
    FoodPriceForecastPromptBuilder,
    build_food_price_agent_config,
)

config = build_food_price_agent_config(model="gemini-3-flash-preview")  # identity
predictor = AgentPredictor(                                              # role
    config,
    FoodPriceForecastPromptBuilder(),
    output_schema=ContinuousAgentForecastOutput,
)
```

### News search and leakage

`enable_news_search=True` attaches a `context_agent` tool — a bounded Google
Search sub-agent.  **Do not use on historical `as_of` dates without accepting
leakage risk**: the search tool has no way to filter to pre-cutoff results.
The search-on predictors are included deliberately to measure how much news
search helps or hurts calibration on historical origins.

### Smoke test

The next cell runs a single `predict()` call as a pre-flight check before the
full backtest.  If `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` are set in
`.env`, a trace URL is printed — useful for inspecting tool calls
(`context_agent`, `set_model_response`) and the structured output.


In [ ]:
import pandas as pd
from aieng.forecasting.methods import ContinuousLLMPredictor, ContinuousLLMPredictorConfig
from food_price_forecasting.analyst_agent import build_food_price_agent_predictor

# ── Baselines ─────────────────────────────────────────────────────────────────
lv = LastValuePredictor()
arima = DartsAutoARIMAPredictor()

# ── LLM predictors (direct structured output, no tools) ───────────────────────
llmp_flash3 = ContinuousLLMPredictor(
    ContinuousLLMPredictorConfig(model="gemini/gemini-3-flash-preview", n_samples=3)
)
llmp_flash3_5 = ContinuousLLMPredictor(
    ContinuousLLMPredictorConfig(model="gemini/gemini-3.5-flash-preview", n_samples=3)
)

# ── Agent predictors — news search OFF (safe for historical backtests) ─────────
agent_flash3 = build_food_price_agent_predictor(
    model="gemini-3-flash-preview", enable_news_search=False
)
agent_flash3_5 = build_food_price_agent_predictor(
    model="gemini-3.5-flash-preview", enable_news_search=False
)

# ── Agent predictors — news search ON (leakage risk on historical as_of dates) ─
agent_flash3_search = build_food_price_agent_predictor(
    model="gemini-3-flash-preview", enable_news_search=True
)
agent_flash3_5_search = build_food_price_agent_predictor(
    model="gemini-3.5-flash-preview", enable_news_search=True
)

all_predictors = [
    lv, arima,
    llmp_flash3, llmp_flash3_5,
    agent_flash3, agent_flash3_5,
    agent_flash3_search, agent_flash3_5_search,
]

# Colors: gray/blue = baselines, red = LLMP, orange = agent no-search, green = agent search
PREDICTOR_COLORS: dict[str, str] = {
    lv.predictor_id:                    "#7f7f7f",
    arima.predictor_id:                 "#1f77b4",
    llmp_flash3.predictor_id:           "#d62728",
    llmp_flash3_5.predictor_id:         "#e87070",
    agent_flash3.predictor_id:          "#ff7f0e",
    agent_flash3_5.predictor_id:        "#ffb347",
    agent_flash3_search.predictor_id:   "#2ca02c",
    agent_flash3_5_search.predictor_id: "#72c472",
}

for p in all_predictors:
    print(f"  {p.predictor_id}")


In [ ]:
# Agent smoke: one predict() call, plain-text summary + Langfuse trace link.
# Requires GEMINI_API_KEY. Langfuse keys optional (LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY).
import logging
from datetime import datetime

from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.langfuse_tracing import init_langfuse_tracing, print_langfuse_trace_url
from food_price_forecasting.analyst_agent import (
    FoodPriceForecastPromptBuilder,
    build_food_price_agent_predictor,
)
from food_price_forecasting.data import build_food_cpi_service
from food_price_forecasting.smoke_report import CFPR_HORIZONS, summarize_agent_predictions
from pydantic import ValidationError

logging.basicConfig(level=logging.WARNING, format="%(message)s")

init_langfuse_tracing()

_AGENT_SMOKE_ORIGIN = datetime(2023, 7, 1)
_AGENT_SMOKE_TASK = ForecastingTask(
    task_id="meat_cfpr",
    target_series_id="cpi_meat_canada",
    horizons=CFPR_HORIZONS,
    frequency="MS",
    description="Meat CFPR; Jan–Dec trajectory from July origin.",
)

ENABLE_NEWS_SEARCH_SMOKE = True  # set False to skip news search (faster, no leakage risk)

_smoke_svc = build_food_cpi_service(cache_dir=STATCAN_CACHE)
_smoke_predictor = build_food_price_agent_predictor(
    model="gemini-3-flash-preview",
    enable_news_search=ENABLE_NEWS_SEARCH_SMOKE,
    prompt_builder=FoodPriceForecastPromptBuilder(max_history_rows=60),
)

print(f"smoke predictor: context_agent={'enabled' if ENABLE_NEWS_SEARCH_SMOKE else 'disabled'}")

_ctx = _smoke_svc.context(as_of=_AGENT_SMOKE_ORIGIN)
try:
    _smoke_preds = _smoke_predictor.predict(_AGENT_SMOKE_TASK, _ctx)
except ValidationError:
    print("\n✗ SCHEMA VALIDATION FAILED")
    print("  The model response did not match the structured forecast schema.")
    print("  See WARNING log above for the raw agent response.")
    _smoke_preds = []

summarize_agent_predictions(_smoke_preds, expected_horizons=CFPR_HORIZONS)
print_langfuse_trace_url()  # flushes spans and prints trace URL (or project page if id unavailable)

---
## 5. Backtest (cached on disk)

`cached_multi_backtest` writes each `BacktestResult` to
`data/predictions/<spec_id>/<predictor_id>/<task_id>.yaml` and reuses it on
subsequent runs.  Pass `force_refresh=True` to re-run a predictor from scratch.

All predictors run against the configured backtest spec (set in the config cell above).
`ContinuousLLMPredictor` and the agent predictor make API calls on a first run
(~6 / ~54 / ~144 calls for `mini_single` / `mini_recent` / `full` respectively);
subsequent runs are free from cache.


In [ ]:
results_by_predictor: dict[str, dict[str, object]] = {}

for predictor in all_predictors:
    print(f"Running {predictor.predictor_id} ...", flush=True)
    results_by_predictor[predictor.predictor_id] = cached_multi_backtest(
        predictor=predictor,
        spec=backtest_spec,
        data_service=svc,
        store_dir=PREDICTIONS_DIR,
    )
    for task_id, result in results_by_predictor[predictor.predictor_id].items():
        print(f"  {task_id:42s}  mean CRPS = {result.mean_crps:.4f}  ({len(result.predictions)} preds)")

---
## 6. Trajectories — overall food CPI

For the overall Canadian food CPI, show the three most recent origins with each
predictor's 12-step trajectory fan.  Solid black is observed history, dashed
black is the Y+1 actuals where available, fans are the predictor's 90%/50%
intervals with the median in colour.

In [ ]:
FOCAL_TASK = "food_cpi_overall_cfpr"
FOCAL_SERIES = "cpi_food_canada"

fig, _ = plot_trajectory_fan(
    results_by_predictor=results_by_predictor,
    task_id=FOCAL_TASK,
    category_id=FOCAL_SERIES,
    data_service=svc,
    n_recent=3,
    colors=PREDICTOR_COLORS,
)
plt.show()

---
## 7. Avg/avg YoY — food CPI categories

The headline CFPR metric: for each July origin, mean predicted CPI for year Y+1
divided by mean observed CPI for year Y, minus 1.  Actual realised YoY (solid
black) is the out-of-sample truth for every completed year.

**Note on the 2022 spike:** LLMP with `reasoning_effort="disable"` anchors its
extrapolation on the *current level* rather than the *rate of change*, so it
tends to underestimate carry-through during a mid-surge origin.  This is a
known limitation of level-domain direct prompting (Gruver / CiK), not a
data-feeding issue.


In [ ]:
from datetime import datetime, timezone


yoy_by_predictor_by_task: dict[str, dict[str, object]] = {}
task_to_category: dict[str, str] = {task.task_id: task.target_series_id for task in backtest_spec.tasks}

_as_of = datetime.now(tz=timezone.utc).replace(tzinfo=None)

for pid, task_results in results_by_predictor.items():
    yoy_by_predictor_by_task[pid] = {}
    for task_id, result in task_results.items():
        actual_df = svc.get_series(result.spec.task.target_series_id, as_of=_as_of)
        yoy_by_predictor_by_task[pid][task_id] = compute_avgyoy(result, actual_df)

fig, _ = plot_avgyoy_grid(
    yoy_by_predictor_by_task=yoy_by_predictor_by_task,
    task_to_category=task_to_category,
    colors=PREDICTOR_COLORS,
)
plt.show()

---
## 8. Model selection

### 8.1 CRPS per category

Lower is better.  The `MEAN` row is the across-category average — useful
context, but the per-category rows are the basis for model selection.

In [ ]:
crps_board = summarize_crps(results_by_predictor)
print(crps_board.to_string())

### 8.2 MAPE per category

Median-accuracy sanity check (CRPS is the primary selection metric).  One panel
per sub-index; each box spans the distribution of per-prediction absolute
percentage errors across all backtest origins and horizons.

In [ ]:
mape_df = compute_mape(results_by_predictor, data_service=svc)
print(mape_df.to_string())

ape_long = compute_ape_long(results_by_predictor, data_service=svc)
fig, _ = plot_mape_by_category(ape_long, task_to_category=task_to_category, colors=PREDICTOR_COLORS)
plt.show()

---
## 9. Backtest-average avg/avg YoY — headline table

Model selection is done **per category**: for each food CPI sub-index the
predictor with the lowest mean CRPS over all backtest origins and horizons for
that category is selected independently.  The table below shows each
category's best predictor and its avg/avg YoY central estimate and uncertainty
band averaged across the full backtest window.

In [ ]:
# Best predictor per category, selected by that category's own mean CRPS.
best_pid_by_task: dict[str, str] = crps_board.drop(index="MEAN").idxmin(axis=1).to_dict()
print("Best predictor by category (mean CRPS over full backtest window):")
for task_id, pid in best_pid_by_task.items():
    category = CATEGORY_LABELS.get(task_to_category[task_id], task_id)
    print(f"  {category:<40s} {pid}")

rows: list[dict[str, object]] = []
for task_id, pid in best_pid_by_task.items():
    yoy_df = yoy_by_predictor_by_task[pid].get(task_id)
    if yoy_df is None or yoy_df.empty:
        continue
    avg = yoy_df[["yoy_median", "yoy_q05", "yoy_q25", "yoy_q75", "yoy_q95", "actual_yoy"]].mean()
    rows.append(
        {
            "category": CATEGORY_LABELS.get(task_to_category[task_id], task_id),
            "best_predictor": pid,
            "median_yoy_%": round(avg["yoy_median"] * 100, 2),
            "q05_%": round(avg["yoy_q05"] * 100, 2),
            "q25_%": round(avg["yoy_q25"] * 100, 2),
            "q75_%": round(avg["yoy_q75"] * 100, 2),
            "q95_%": round(avg["yoy_q95"] * 100, 2),
            "actual_yoy_%": round(avg["actual_yoy"] * 100, 2),
        }
    )

headline = pd.DataFrame(rows).set_index("category")
print()
print(headline.to_string())

---
## 10. Protected evaluation

`EvalTracker` enforces `max_runs` from the YAML spec against a shared
`data/eval_runs.yaml` file.  Run this only when you believe the predictor
is ready for a final assessment — each call spends one of your five runs.

In [ ]:
if _EVAL_SPEC_FILE is None:
    print("Protected evaluation is not available for mini configs. Switch EXPERIMENT_CONFIG to 'full' to use it.")
else:
    with (SPECS_DIR / _EVAL_SPEC_FILE).open() as f:
        eval_spec = MultiTargetEvalSpec.model_validate(yaml.safe_load(f))

    print(describe_spec(eval_spec, data_service=svc))

    tracker = EvalTracker(EVAL_RUNS_PATH)
    used = tracker.runs_for(eval_spec.spec_id)
    print(f"\nBudget: {used} / {eval_spec.max_runs} runs used for spec '{eval_spec.spec_id}'.")


In [ ]:
# ── Uncomment to spend a run ──────────────────────────────────────────────────
# from aieng.forecasting.evaluation.eval import multi_evaluate
#
# best_predictor = next(p for p in predictors if p.predictor_id == best_pid)
# eval_results = multi_evaluate(
#     predictor=best_predictor,
#     spec=eval_spec,
#     data_service=svc,
#     tracker=tracker,
# )
# for task_id, r in eval_results.items():
#     print(f"  {task_id:42s}  mean CRPS = {r.mean_crps:.4f}  (run {r.run_number}/{eval_spec.max_runs})")